In [1]:
import sys
import os
import time as _time
import pandas as pd
import inspect
from pathlib import Path


 
# Find the project root
def find_project_root(marker="src/data_prep.py"):

    # Calls the class working directory from pathlib, and captures the starting location
    here = Path.cwd()

    # Creates an upward search path, checking one folder layer at a time until
    # candidate (the project root) is returned 
    for candidate in [here, *here.parents]:
        if (candidate / marker).exists():
            return candidate
        
    # Incase the file is not found
    raise FileNotFoundError("Could not find the file")

# Establishing the project paths and root
project_root = find_project_root()
src_path = project_root / "src"
data_path = project_root / "data"



print(f"Project root confirmed at: {project_root}")

# Insert src directory at top priority
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

# Import the processing function
from data_prep import build_full_dataset
from data_prep import fetch_sequence
from data_prep import build_ClinVar_dataset 

#Sanity checks to see if the function is actually returning the DNA sequence
print(fetch_sequence("14", 23412740))       


ClinVar_path = str(data_path / "raw" / "variant_summary.txt.gz")

#start = _time.time()
#test_df = build_ClinVar_dataset(ClinVar_path, "test_out.csv", limit=500)
#elapsed = _time.time() - start

#print(f"{elapsed:.1f}s for {len(test_df)} rows kept")

# Defining the ClinVar file path (includes lots of variants, which are filtered in data_prep)


# Unlike ClinVar, gnomAD includes seperated data for each gene. This takes
# the paths for each of the three genes.
gnomAD_csv_paths = {
    "MYH7": str(data_path / "raw" / "gnomAD_MYH7.csv"),
    "MYBPC3": str(data_path / "raw" / "gnomAD_MYBPC3.csv"),
    "TTN": str(data_path / "raw" / "gnomAD_TTN.csv")
}
out_path = str(data_path / "processed" / "dataset.csv")

# Runs data preparation pipeline and calls build_ClinVar_dataset from data_prep
# note that build_full_dataset was originally ran, but for the second time running, because
# I already loaded all the benign gnomAD variants, only build_ClinVar_dataset is neccesary
print("Starting data preparation pipeline")

final_df = build_ClinVar_dataset(
    ClinVar_path=ClinVar_path,
    out_path=out_path,
    old_frac=0.5,   # this part was already saved
    new_frac=1.0,   # new parts that need to be added
)

# Configure Pandas display options and show results
pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)
pd.set_option("display.max_colwidth", None)





print("\n--- ENTIRE FINISHED DATASET ---")
display(final_df)

Project root confirmed at: c:\Users\jaira\Desktop\Official cardiac project\Cardiac-project
CCTTTGAATTTGCCCCCACTCCCCTGCATTTCCACCTCCCCTAGTCCAGACCCCATCCCCGGGTCCTCAGGGCAGTGAAGAAGAGTCTGGAGTCAGGATCTCGGCTTCAAGGAAAATTGCTTTATTCTGCTTCCTCCCAAGGAGCTGTTACACAGGCTCCAGCATGGGGCTTTGCTGGCACCTCCAGGGCTGAGCAGATCA
Starting data preparation pipeline


KeyboardInterrupt: 

In [ ]:
import os
!git clone https://github.com/Naman-Muthreja/Cardiac-project.git
%cd Cardiac-project
!git pull
%cd src

import torch
print("GPU is available", torch.cuda.is_available())

import sys
from pathlib import Path
import pandas as pd

def find_project_root(marker="src/data_prep.py"):
    here = Path.cwd()
    for candidate in [here, *here.parents]:
        if (candidate/marker).exists():
            return candidate
    raise FileNotFoundError("Could not find the file")

project_root = find_project_root()
src_path = project_root/"src"
data_path = project_root/"data"

if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

from train import train_model

dataset_path = data_path/"processed"/"dataset.csv"
df = pd.read_csv(dataset_path)
print(df.shape)

model, (X_test, y_test), demo_df = train_model(df, evaluate_test=True)

# Saving immediately after
torch.save(model.state_dict(), str(project_root / "model_state.pt"))
demo_df.to_csv(str(project_root / "demo_holdout.csv"), index=False)

os.chdir(str(project_root))
!git add model_state.pt demo_holdout.csv
!git commit -m "Add trained model weights and demo set"
!git push

Cloning into 'Cardiac-project'...
remote: Enumerating objects: 191, done.
remote: Counting objects: 100% (191/191), done.
remote: Compressing objects: 100% (119/119), done.
remote: Total 191 (delta 118), reused 136 (delta 63), pack-reused 0 (from 0)
Receiving objects: 100% (191/191), 1.30 MiB | 19.29 MiB/s, done.
Resolving deltas: 100% (118/118), done.
/content/Cardiac-project/Cardiac-project
Already up to date.
/content/Cardiac-project/Cardiac-project/src
GPU is available True
(12552, 8)
Benign capped: 7155
Counts of each class in the training: {'HCM': np.int64(311), 'DCM': np.int64(662), 'Benign': np.int64(3891)}
Evaluation has started for the training predictions
Epoch 1/25 | Train Loss: 0.8573 | Val Acc: 47.2397%
Evaluation has started for the training predictions
Epoch 2/25 | Train Loss: 0.7204 | Val Acc: 30.8875%
Evaluation has started for the training predictions
Epoch 3/25 | Train Loss: 0.6648 | Val Acc: 65.8980%
Evaluation has started for the training predictions
Epoch 4/25 | 

In [4]:
import pandas as pd
from sklearn.model_selection import train_test_split
from train import cap_benign 
import annotate

data_path = project_root/"data"
dataset_path = data_path/"processed"/"dataset.csv"
# Reads the dataset_path, same seed for reproducibility
df = pd.read_csv(dataset_path)
print(dataset_path)
seed = 42

# Calls cap_benign from train to cap the benign variants amount
capped = cap_benign(df, max_benign=None, seed=seed)

# Makes the datasets also inside of train, uses data stratification
rest_df, demo_df = train_test_split(capped, test_size= 0.02, stratify=capped["label"], random_state=seed)
train_df, test_df = train_test_split(rest_df, test_size=0.10/0.98, stratify=rest_df["label"], random_state=seed)

# Prints the value count of the amount of rows and each label
print(f"test_df = {len(test_df)} rows")
print(test_df["label"].value_counts())

# Calls build_annotation_subset from annotate.py
subset = annotate.build_annotation_subset(df, test_df)

# Calls annotate_dataset from annotate.py, with no in_path, and a csv output. 
annotated = annotate.annotate_dataset (None, str(data_path / "processed" / "dataset_annotated.csv"), df=subset)

results = annotate.revel_cadd_benchmark(annotated, test_df)

# Returns the results by using .items() to return a (key, value) pair each time
print("\n--- Binary Pathogenic-vs-Benign AUC-ROC, same 716 test variants ---")
for name, auc in results.items():
    print(f"{name:35s} {auc:.3f}")


c:\Users\jaira\Desktop\Official cardiac project\Cardiac-project\data\processed\dataset.csv
Benign capped: 7155
test_df = 716 rows
label
Benign    573
DCM        97
HCM        46
Name: count, dtype: int64
Benign capped: 7155
Kept 23.4 % of train_pool variants, which is 6418
Annotation subset: 2216 rows (716 to test on + 1500 to train with)
CADD Coverage: 100.0%
REVEL Coverage: 7.4%
REVEL coverage by class:
label
Benign     5.6%
DCM        0.0%
HCM       45.1%
Name: revel_scores, dtype: str
CADD coverage by class:
label
Benign    100.0%
DCM       100.0%
HCM       100.0%
Name: cadd_phred, dtype: str


AttributeError: 'NoneType' object has no attribute 'drop_duplicates'

In [ ]:
# --- SETUP: run this once per kernel ---
import sys
from pathlib import Path
import pandas as pd

def find_project_root(marker="src/data_prep.py"):
    here = Path.cwd()
    for candidate in [here, *here.parents]:
        if (candidate / marker).exists():
            return candidate
    raise FileNotFoundError("Could not find the project root")

project_root = find_project_root()
src_path  = project_root / "src"
data_path = project_root / "data"

if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

print("project root :", project_root)
print("src on path  :", str(src_path) in sys.path)
print("consequence.py exists:", (src_path / "consequence.py").exists())

# --- STEP 1g CHECK ---
from consequence import parse_consequence

dataset_path = data_path / "processed" / "dataset.csv"
df = pd.read_csv(dataset_path)
df["consequence"] = df["name"].apply(parse_consequence)

print("\n--- COUNTS ---")
print(pd.crosstab(df["consequence"], df["label"]))
print("\n--- FRACTION OF EACH CLASS ---")
print(pd.crosstab(df["consequence"], df["label"], normalize="columns").round(3))

file : c:\Users\jaira\Desktop\Official cardiac project\Cardiac-project\src\consequence.py
size on disk : 0 bytes
line count   : 0
has 'def parse_consequence' : False
cached in kernel : YES
names it defines : []

--- FIRST 25 LINES AS SAVED ON DISK ---
    (FILE IS EMPTY)


In [2]:
from model import CardiacCNN
import torch

# Build a fresh, untrained model with the same architecture
loaded_model = CardiacCNN(seq_len=201, n_classes=3)

# Load the saved weights from Drive into it
loaded_model.load_state_dict(torch.load("/content/drive/MyDrive/model_state_backup_0.954.pt"))
loaded_model.eval()

print(loaded_model)

KeyboardInterrupt: 